# **SQLAlchemy Tutorial for Beginners**

This notebook teaches SQLAlchemy from the ground up using SQLite and examples similar to a chatbot backend.

By the end, you should understand:
- `engine`
- `Base`
- ORM models
- `SessionLocal`
- insert, select, update, delete
- how chat history and long-term memory can be stored


## 1. What is SQLAlchemy?

SQLAlchemy is a Python library for working with relational databases.

With raw SQL:

```sql
INSERT INTO users (name, age) VALUES ('Alice', 30);
```

With SQLAlchemy ORM:

```python
user = User(name="Alice", age=30)
db.add(user)
db.commit()
```

ORM means **Object Relational Mapper**:

```text
Python class  <-> database table
Python object <-> database row
Python field  <-> database column
```


## 2. Install

```bash
pip install sqlalchemy
```


In [1]:
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime
from pathlib import Path

print("Imports successful")


Imports successful


## 3. Create the engine

The engine is SQLAlchemy's connection interface to your database.


In [2]:
Path("data").mkdir(exist_ok=True)

DATABASE_URL = "sqlite:///data/sqlalchemy_tutorial.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

print(engine)


Engine(sqlite:///data/sqlalchemy_tutorial.db)


`sqlite:///data/sqlalchemy_tutorial.db` means the database will be saved as:

```text
data/sqlalchemy_tutorial.db
```

`check_same_thread=False` is commonly used with SQLite in web apps where multiple threads may access the database.


## 4. Create `Base`

`Base` is the parent class for your ORM models.


In [3]:
Base = declarative_base()
print(Base)


<class 'sqlalchemy.orm.decl_api.Base'>


## 5. Define a table as a Python class


In [4]:
class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    email = Column(String, unique=True, index=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    def __repr__(self):
        return f"User(id={self.id}, name={self.name!r}, email={self.email!r})"


This roughly corresponds to:

```sql
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    email VARCHAR UNIQUE,
    created_at DATETIME
);
```

Important ideas:
- `__tablename__` = database table name
- `primary_key=True` = unique row identifier
- `unique=True` = duplicate values are not allowed
- `index=True` = speeds up lookups
- `nullable=False` = value is required


## 6. Create the table


In [5]:
Base.metadata.create_all(bind=engine)
print("Tables created")


Tables created


## 7. Create a session

A session is your working connection to the database.

Typical pattern:

```text
open session
   ↓
query / add / update
   ↓
commit
   ↓
close
```


In [ ]:
SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)

db = SessionLocal()

print(db)
db.close()


## 8. INSERT data


In [7]:
db = SessionLocal()

user = User(
    name="Alice",
    email="alice@example.com"
)

db.add(user)
db.commit()
db.refresh(user)

print(user)

db.close()


User(id=1, name='Alice', email='alice@example.com')


Meaning:

```python
db.add(user)
```
adds the object to the transaction.

```python
db.commit()
```
saves it to the database.

```python
db.refresh(user)
```
reloads generated values such as `id`.


## 9. SELECT data


In [8]:
db = SessionLocal()

users = db.query(User).all()

for user in users:
    print(user)

db.close()


User(id=1, name='Alice', email='alice@example.com')


In [9]:
db = SessionLocal()

user = (
    db.query(User)
    .filter(User.email == "alice@example.com")
    .first()
)

print(user)

db.close()


User(id=1, name='Alice', email='alice@example.com')


Read this:

```python
db.query(User)
```

as roughly:

```sql
SELECT * FROM users;
```

and:

```python
.filter(User.email == "alice@example.com")
```

as:

```sql
WHERE email = 'alice@example.com'
```


## 10. UPDATE data


In [10]:
db = SessionLocal()

user = (
    db.query(User)
    .filter(User.email == "alice@example.com")
    .first()
)

if user:
    user.name = "Alice Smith"
    db.commit()
    db.refresh(user)

print(user)

db.close()


User(id=1, name='Alice Smith', email='alice@example.com')


You update the Python object, then call `commit()`. SQLAlchemy tracks the change.


## 11. DELETE data


In [ ]:
db = SessionLocal()

temp_user = User(
    name="Temporary User",
    email="temporary@example.com"
)

db.add(temp_user)
db.commit()

db.delete(temp_user)
db.commit()

print("Temporary user deleted")

db.close()


## 12. Why `try ... finally`?

A safe database function often looks like this:

```python
db = SessionLocal()

try:
    ...
finally:
    db.close()
```

This guarantees the session closes even if an error occurs.


In [ ]:
def get_all_users():
    db = SessionLocal()

    try:
        return db.query(User).all()
    finally:
        db.close()

print(get_all_users())


# Part 2: Chatbot-style example

Now create three tables:

```text
Conversation
ChatMessage
LongTermMemory
```


In [11]:
class Conversation(Base):
    __tablename__ = "tutorial_conversations"

    id = Column(Integer, primary_key=True, index=True)
    thread_id = Column(String, unique=True, index=True)
    title = Column(String, default="New Chat")
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow)


class ChatMessage(Base):
    __tablename__ = "tutorial_chat_messages"

    id = Column(Integer, primary_key=True, index=True)
    thread_id = Column(String, index=True)
    role = Column(String)
    content = Column(Text)
    created_at = Column(DateTime, default=datetime.utcnow)


class LongTermMemory(Base):
    __tablename__ = "tutorial_long_term_memory"

    id = Column(Integer, primary_key=True, index=True)
    thread_id = Column(String, index=True)
    memory = Column(Text)
    created_at = Column(DateTime, default=datetime.utcnow)


Base.metadata.create_all(bind=engine)

print("Chatbot tables created")


Chatbot tables created


## 13. Create a conversation


In [12]:
def create_conversation(thread_id: str, title: str = "New Chat"):
    db = SessionLocal()

    try:
        conversation = Conversation(
            thread_id=thread_id,
            title=title
        )

        db.add(conversation)
        db.commit()
        db.refresh(conversation)

        return conversation

    finally:
        db.close()


In [13]:
conversation = create_conversation(
    thread_id="thread_001",
    title="Learning SQLAlchemy"
)

print(conversation.id, conversation.thread_id, conversation.title)


1 thread_001 Learning SQLAlchemy


## 14. Save chat messages


In [14]:
def save_message(thread_id: str, role: str, content: str):
    db = SessionLocal()

    try:
        message = ChatMessage(
            thread_id=thread_id,
            role=role,
            content=content
        )

        db.add(message)
        db.commit()
        db.refresh(message)

        return message

    finally:
        db.close()


In [15]:
save_message("thread_001", "user", "What is SQLAlchemy?")
save_message(
    "thread_001",
    "assistant",
    "SQLAlchemy is a Python toolkit and ORM for relational databases."
)

print("Messages saved")


Messages saved


## 15. Load chat history


In [16]:
def get_chat_history(thread_id: str):
    db = SessionLocal()

    try:
        return (
            db.query(ChatMessage)
            .filter(ChatMessage.thread_id == thread_id)
            .order_by(ChatMessage.created_at.asc())
            .all()
        )
    finally:
        db.close()


In [17]:
history = get_chat_history("thread_001")

for message in history:
    print(f"{message.role}: {message.content}")


user: What is SQLAlchemy?
assistant: SQLAlchemy is a Python toolkit and ORM for relational databases.


This query means:

```text
Query ChatMessage
where thread_id matches
sort oldest -> newest
return all rows
```


## 16. Save long-term memory


In [18]:
def save_memory(thread_id: str, memory: str):
    db = SessionLocal()

    try:
        item = LongTermMemory(
            thread_id=thread_id,
            memory=memory
        )

        db.add(item)
        db.commit()

    finally:
        db.close()


In [19]:
save_memory(
    "thread_001",
    "The user is learning SQLAlchemy."
)

print("Memory saved")


Memory saved


## 17. Retrieve recent memories


In [20]:
def get_recent_memories(thread_id: str, limit: int = 20):
    db = SessionLocal()

    try:
        return (
            db.query(LongTermMemory)
            .filter(LongTermMemory.thread_id == thread_id)
            .order_by(LongTermMemory.created_at.desc())
            .limit(limit)
            .all()
        )
    finally:
        db.close()


In [21]:
memories = get_recent_memories("thread_001")

for item in memories:
    print("-", item.memory)


- The user is learning SQLAlchemy.


# 18. Mental model

Remember this order:

```text
Engine
  ↓
How do I connect?

Base
  ↓
What classes become tables?

Models
  ↓
What columns exist?

SessionLocal
  ↓
How do I open a database session?

Query / Add / Update / Delete
  ↓
What operation do I want?

commit()
  ↓
Save changes

close()
  ↓
Release the session
```

The key idea:

> SQLAlchemy ORM lets you work with database rows as Python objects.


# 19. Cheat sheet

| Task | SQLAlchemy |
|---|---|
| Open session | `db = SessionLocal()` |
| Add row | `db.add(obj)` |
| Save | `db.commit()` |
| Refresh | `db.refresh(obj)` |
| Query table | `db.query(Model)` |
| Filter | `.filter(Model.field == value)` |
| First row | `.first()` |
| All rows | `.all()` |
| Sort | `.order_by(...)` |
| Limit | `.limit(10)` |
| Delete | `db.delete(obj)` |
| Close | `db.close()` |


# 20. Exercises

Try these without copying earlier cells:

1. Create a `Product` table with `id`, `name`, and `price`.
2. Insert three products.
3. Query all products.
4. Find one product by name.
5. Update its price.
6. Delete one product.
7. Create a new chatbot conversation and save three messages under the same `thread_id`.

If you can do those, you understand the core SQLAlchemy workflow.


# 21. Mapping this to your chatbot

Your chatbot follows the same architecture:

```text
engine
   ↓
SQLite connection

SessionLocal
   ↓
creates sessions

Base
   ↓
parent ORM class

Conversation / ChatMessage / LongTermMemory
   ↓
database tables

save_chat_message() / get_chat_history() / save_memory()
   ↓
CRUD operations
```

Once this notebook makes sense, your project's `database.py` should become much easier to read.
